In [54]:
print("Hello, World!" )

Hello, World!

In [55]:
import warnings,os
from dotenv import load_dotenv

load_dotenv()

True

In [56]:
assert os.environ.get("OPENROUTER_API_KEY") is not None, "OPENROUTER_API_KEY is not set in the environment variables."

In [57]:
import langchain.mcp

MCP Adapter in LangChain

In [58]:
%%writefile cinebot_mcp_server.py
from fastmcp import FastMCP
from mcp.types import ToolAnnotations

mcp = FastMCP("CineBot")

@mcp.tool()
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie."""
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found.")

@mcp.tool(annotations=ToolAnnotations(destructiveHint=True))
def cancel_booking(booking_id: str) -> str:
    """Cancel an existing booking. Irreversible."""
    return f"Booking {booking_id} cancelled."

@mcp.tool()
def get_seat_map(movie_title: str) -> dict:
    """Get the seat map for a movie -- returns structured data, not just text."""
    return {"movie": movie_title, "available_rows": ["A", "B", "C"], "sold_out_rows": ["D"]}

if __name__ == "__main__":
    mcp.run()

Overwriting cinebot_mcp_server.py


In [59]:
from pathlib import Path
from langchain.mcp import MCPAdapter
from fastmcp.client.transports import PythonStdioTransport

In [60]:
cinebot_mcp_transport = PythonStdioTransport(
    script_path=Path("C:\AgenticAI\MCP\Live-Class-2026-main\Weekend 14 - Oct 3 & Oct 4\MCP in Langchain\cinebot_mcp_server.py"),
    log_file=Path("cinebot_mcp_server.log")
)

async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    # Now you can use the mcp_adapter to interact with the MCP server
    tools=await mcp_adapter.list_tools()
    print("Available tools:", [t.name for t in tools])

<>:2: SyntaxWarning: invalid escape sequence '\A'
C:\Users\mohsinurrahman\AppData\Local\Temp\ipykernel_33612\4222889100.py:2: SyntaxWarning: invalid escape sequence '\A'
  script_path=Path("C:\AgenticAI\MCP\Live-Class-2026-main\Weekend 14 - Oct 3 & Oct 4\MCP in Langchain\cinebot_mcp_server.py"),


Available tools:
['check_showtimes', 'cancel_booking', 'get_seat_map']

In [61]:
from langchain.agents import create_agent

Call a tool without any AI or Agent

In [62]:
from rich import print

In [79]:
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

model = init_chat_model(
    model="openrouter/free",
    model_provider="openrouter",
)


tool_call_result=''
async with MCPAdapter(cinebot_mcp_transport) as adapter:
    tools = await adapter.list_tools()

    showtime_tool = next((t for t in tools if t.name == "check_showtimes"), None)

    direct_result = await showtime_tool.ainvoke({"movie_title": "interstellar"})
    print("Direct invocation result:", direct_result)

# Calling with Agent as well




    try:
        cinebot_agent = create_agent(model=model,tools=tools)
        result = await cinebot_agent.ainvoke({"messages":[('user', "What are the showtimes for Interstellar?")]})
        tool_call_result = result
        print("Agent invocation result:", result)
    except Exception as e:
        print("Error during agent invocation:", e)

Direct invocation result:
[{'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_72e709e4-f0e1-4af4-97be-0c4a4a9f05a4'}]

Agent invocation result:
{
    'messages': [
        HumanMessage(
            content='What are the showtimes for Interstellar?',
            additional_kwargs={},
            response_metadata={},
            id='43fd119a-cc0c-457e-b9ae-29dc6a741fe5'
        ),
        AIMessage(
            content='I\'ll check the showtimes for "Interstellar" for you.',
            additional_kwargs={},
            response_metadata={
                'model_name': 'poolside/laguna-s-2.1:free',
                'id': 'gen-1791055816-xIn3VeLfqYCRImYiLRAb',
                'created': 1791055816,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--01a1033e-e309-7733-9f59-1b9bdf9a1791-0',
            tool_calls=[
                {
                    'name': 'check_showtimes',
                    'args': {'movie_title': 'Interstellar'},
                    'id': 'chatcmpl-tool-c5205eb8b5544160b49d80190efb5634',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 293,
                'output_tokens': 39,
                'total_tokens': 332,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content=[
                {'type': 'text', 'text': '7:00 PM and 10:15 PM', 'id': 'lc_c72a4b50-8522-4fca-8c22-1506289be826'}
            ],
            name='check_showtimes',
            id='534aa9cc-cf2b-4d57-8fbc-74eb106e89cf',
            tool_call_id='chatcmpl-tool-c5205eb8b5544160b49d80190efb5634',
            artifact={'structured_content': {'result': '7:00 PM and 10:15 PM'}}
        ),
        AIMessage(
            content='The showtimes for **Interstellar** are:\n- **7:00 PM**\n- **10:15 PM**',
            additional_kwargs={
                'reasoning_content': 'The user asked for showtimes for Interstellar, and I got the result: "7:00 PM
and 10:15 PM". I should provide this information to the user in a clear format.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.text',
                        'format': 'unknown',
                        'index': 0,
                        'text': 'The user asked for showtimes for Interstellar, and I got the result: "7:00 PM and 
10:15 PM". I should provide this information to the user in a clear format.'
                    }
                ]
            },
            response_metadata={
                'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free',
                'id': 'gen-1791055824-Iqc8kzyn4JJMDYVayF8w',
                'created': 1791055824,
                'object': 'chat.completion',
                'finish_reason': 'stop',
                'logprobs': None,
                'model_provider': 'openrouter',
                'cost': 0.0,
                'cost_details': {
                    'upstream_inference_completions_cost': 0.0,
                    'upstream_inference_prompt_cost': 0.0,
                    'upstream_inference_cost': 0.0
                }
            },
            id='lc_run--01a1033f-01b0-7f01-ac2a-61bcc567b44a-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 519,
                'output_tokens': 75,
                'total_tokens': 594,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 40}
           

In [64]:
tool_call_result['messages'][-1]

AIMessage(content='The showtimes for Interstellar are:\n- 7:00 PM\n- 10:15 PM', additional_kwargs={'reasoning_content': 'Okay, the function returned the showtimes for Interstellar. The response indicates that the movie has showtimes at 7:00 PM and 10:15 PM. I should present this information clearly to the user.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Okay, the function returned the showtimes for Interstellar. The response indicates that the movie has showtimes at 7:00 PM and 10:15 PM. I should present this information clearly to the user.'}]}, response_metadata={'model_name': 'cohere/north-mini-code:free', 'id': 'gen-1791055321-WhVX8veAyu2bdfnUV8DS', 'created': 1791055321, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01

Structured Output

In [81]:
async with MCPAdapter(cinebot_mcp_transport) as adapter:
    tools = await adapter.list_tools()
    seat_map_tool = next(t for t in tools if t.name == "get_seat_map")

    tool_call={
        'name':'get_seat_map',
        'args':{'movie_title':'Dune Part Two'},
        'id':'tool_call_id_12345',
        'type':'tool_call'

    }

    message=await seat_map_tool.ainvoke(tool_call)
    print('Text Content(what my model reads)',message.content)
    print('Stuctured data(what my code use if required)',message.artifact)
    print('Accessing data directly:',message.artifact['structured_content']['available_rows']) #accessing structured data from the tool call result


     

Text Content(what my model reads)
[
    {
        'type': 'text',
        'text': '{"movie":"Dune Part Two","available_rows":["A","B","C"],"sold_out_rows":["D"]}',
        'id': 'lc_87ab9c27-9280-4ddd-ab81-bea62fdbdba2'
    }
]

Stuctured data(what my code use if required)
{'structured_content': {'movie': 'Dune Part Two', 'available_rows': ['A', 'B', 'C'], 'sold_out_rows': ['D']}}

Accessing data directly:
['A', 'B', 'C']

Error Handling

In [66]:
%%writefile cinebot_error_server.py
from fastmcp import FastMCP
mcp = FastMCP("CineBot")

@mcp.tool()
def risky_lookup(booking_id: str) -> str:
    """Look up a booking --fails if ID format is wrong."""
    if not booking_id.startswith("BK-"):
        raise ValueError(f"Invalid booking ID format:{booking_id!r}. Must start with 'BK-'.")
    return f"Booking {booking_id} confirmed."

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_error_server.py


In [67]:
error_meessage_global=''

error_transport = PythonStdioTransport(
    script_path=Path('cinebot_error_server.py'),
    log_file=Path('cinebot_error_server.log')

)

async with MCPAdapter(error_transport) as adapter:
    tools= await adapter.list_tools()
    risky_lookup_tool=tools[0]  # Assuming there's only one tool in this example

    bad_call={
        'name':'risky_lookup',
        'args':{'booking_id':'12345'},  # Invalid format
        'id':'bad_call_001',
        'type':'tool_call'
    }

    error_message= await risky_lookup_tool.ainvoke(
        bad_call
    )

    print('Error message from tool:', error_message)

    print('content:',error_message.content)
    error_meessage_global=error_message
    print(error_message)





Error message from tool:
ToolMessage(
    content=[
        {
            'type': 'text',
            'text': "Error calling tool 'risky_lookup': Invalid booking ID format:'12345'. Must start with 'BK-'.",
            'id': 'lc_14ca6747-1f61-4665-b9ac-3a67f866935b'
        }
    ],
    name='risky_lookup',
    tool_call_id='bad_call_001',
    status='error'
)

content:
[
    {
        'type': 'text',
        'text': "Error calling tool 'risky_lookup': Invalid booking ID format:'12345'. Must start with 'BK-'.",
        'id': 'lc_14ca6747-1f61-4665-b9ac-3a67f866935b'
    }
]

ToolMessage(
    content=[
        {
            'type': 'text',
            'text': "Error calling tool 'risky_lookup': Invalid booking ID format:'12345'. Must start with 'BK-'.",
            'id': 'lc_14ca6747-1f61-4665-b9ac-3a67f866935b'
        }
    ],
    name='risky_lookup',
    tool_call_id='bad_call_001',
    status='error'
)

In [68]:
error_meessage_global

ToolMessage(content=[{'type': 'text', 'text': "Error calling tool 'risky_lookup': Invalid booking ID format:'12345'. Must start with 'BK-'.", 'id': 'lc_14ca6747-1f61-4665-b9ac-3a67f866935b'}], name='risky_lookup', tool_call_id='bad_call_001', status='error')

Use metadata to put human in the loop

In [69]:
def is_destructive_tool_call(tool):
    # return message.type == "tool_call" and message.name == "cancel_booking"
    meta = (tool.metadata or {}).get('mcp',{}).get('tool',{}).get('annotations',{})
    return bool(meta.get('destructive_hint'))


async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()
    for t in tools:
        print(f"Tool: {t.name}, Destructive: {is_destructive_tool_call(t)}, Tools:{t}")

Tool: check_showtimes, Destructive: False, Tools:name='check_showtimes' description='Check available showtimes for 
a movie.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'movie_title': {'type': 
'string'}}, 'required': ['movie_title']} metadata={'mcp': {'tool': {'_meta': {'fastmcp': {'tags': []}}}, 'server': 
{'name': 'CineBot', 'version': '4.0.10'}}} handle_tool_error=<function _handle_mcp_tool_error at 
0x0000024AA099A840> response_format='content_and_artifact' coroutine=<function as_langchain_tool.<locals>.call_tool
at 0x0000024AA5B1B420>

Tool: cancel_booking, Destructive: True, Tools:name='cancel_booking' description='Cancel an existing booking. 
Irreversible.' args_schema={'type': 'object', 'additionalProperties': False, 'properties': {'booking_id': {'type': 
'string'}}, 'required': ['booking_id']} metadata={'mcp': {'tool': {'annotations': {'destructive_hint': True}, 
'_meta': {'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}} 
handle_tool_error=<function _handle_mcp_tool_error at 0x0000024AA099A840> response_format='content_and_artifact' 
coroutine=<function as_langchain_tool.<locals>.call_tool at 0x0000024AA5BF19E0>

Tool: get_seat_map, Destructive: False, Tools:name='get_seat_map' description='Get the seat map for a movie -- 
returns structured data, not just text.' args_schema={'type': 'object', 'additionalProperties': False, 
'properties': {'movie_title': {'type': 'string'}}, 'required': ['movie_title']} metadata={'mcp': {'tool': {'_meta':
{'fastmcp': {'tags': []}}}, 'server': {'name': 'CineBot', 'version': '4.0.10'}}} handle_tool_error=<function 
_handle_mcp_tool_error at 0x0000024AA099A840> response_format='content_and_artifact' coroutine=<function 
as_langchain_tool.<locals>.call_tool at 0x0000024AA5BF2FC0>

In [70]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

async with MCPAdapter(cinebot_mcp_transport) as mcp_adapter:
    tools= await mcp_adapter.list_tools()

    for t in tools:
        print(f"{t.name}: Destructive: {is_destructive_tool_call(t)}")

    #One HITL config, built dynamically from whatever the server marked destructive

    interrup_on={
        t.name:{
            "allowed_decisions":["approve","reject","Edit"]

        }
        for t in tools
        if is_destructive_tool_call(t)            
    }

    print("\ninterrupt_on config for HITL:",interrup_on)

    hitl_middleware=HumanInTheLoopMiddleware(interrupt_on=interrup_on)
    cinebot_guarded=create_agent(
        model=model,
        tools=tools,
        middleware=[hitl_middleware]
    )
    


check_showtimes: Destructive: False

cancel_booking: Destructive: True

get_seat_map: Destructive: False

interrupt_on config for HITL:
{'cancel_booking': {'allowed_decisions': ['approve', 'reject', 'Edit']}}


9. Elicitation: When the Server Needs to Ask a Question Mid-Call
Some MCP tools need clarification partway through a call — "confirm you want to proceed?". langchain.mcp answers this automatically with a LangGraph interrupt() — every client MCPAdapter builds is armed for it by default, no extra code needed to enable it.

In [71]:
%%writefile cinebot_elicit_server.py
from fastmcp import FastMCP, Context

mcp = FastMCP("CineBotElicit")

@mcp.tool()
async def cancel_booking_confirm(booking_id: str, ctx: Context) -> str:
    """Cancel a booking, but ask the caller to confirm first."""
    result = await ctx.elicit(
        f"Confirm cancellation of booking {booking_id}?",
        response_type=bool,
    )
    if result.action != "accept":
        return f"Cancellation of {booking_id} aborted ({result.action})."
    if not result.data:
        return f"Cancellation of {booking_id} aborted (user said no)."
    return f"Booking {booking_id}: cancelled."

if __name__ == "__main__":
    mcp.run()


Overwriting cinebot_elicit_server.py


In [72]:
elicit_transport = PythonStdioTransport(
    script_path=Path('cinebot_elicit_server.py'),
    log_file=Path('cinebot_elicit_server.log')
)

async with MCPAdapter(elicit_transport) as mcp_adapter:
    tools = await mcp_adapter.list_tools()

    confirm_tool = next(t for t in tools if t.name == "cancel_booking_confirm")

    tool_call={
        'name':'cancel_booking_confirm',
        'args':{'booking_id':'BK12345'},
        'id':'call_1',
        'type':'tool_call'
    }

    elicit_result= await confirm_tool.ainvoke(tool_call)
    print("Status:",elicit_result.status)
    print("Content:",elicit_result.content)

Status: error

Content:
[
    {
        'type': 'text',
        'text': 'elicitation via server-initiated requests is unavailable on 2026-07-28 connections.',
        'id': 'lc_58f761ff-92ed-4f50-9caf-f248d8e5b4bd'
    }
]

Connecting to multiple server

In [85]:
from fastmcp.mcp_config import MCPConfig
import sys
sys.stderr = sys.__stderr__  # real stderr with a fileno()

fleet_config: MCPConfig = {
    "mcpServers": {
        "cinebot": {
            "command": "C:\\AgenticAI\\MCP\\Live-Class-2026-main\\Weekend 14 - Oct 3 & Oct 4\\MCP in Langchain\\.venv\\Scripts\\python.exe",
            "args": ["C:\\AgenticAI\\MCP\\Live-Class-2026-main\\Weekend 14 - Oct 3 & Oct 4\\MCP in Langchain\\cinebot_mcp_server.py"],
        },
        "errors": {
            "command": "C:\\AgenticAI\\MCP\\Live-Class-2026-main\\Weekend 14 - Oct 3 & Oct 4\\MCP in Langchain\\.venv\\Scripts\\python.exe",
            "args": ["C:\\AgenticAI\\MCP\\Live-Class-2026-main\\Weekend 14 - Oct 3 & Oct 4\\MCP in Langchain\\cinebot_error_server.py"],
        },
    }
}

In [86]:
async with MCPAdapter(fleet_config) as adapter:
    fleet_tools = await adapter.list_tools()
    print("MsConfig fleet tools:", [t.name for t in fleet_tools])

[10/03/26 14:41:28] INFO     Proxy detected connected client - reusing existing session for all       proxy.py:1383
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

[10/03/26 14:41:30] INFO     Proxy detected connected client - reusing existing session for all       proxy.py:1383
                             requests. This may cause context mixing in concurrent scenarios, and the              
                             session's existing settings apply, so backend results are validated                   
                             against their declared output schema rather than relayed as-is. Pass a                
                             disconnected client to avoid both.                                                    

MsConfig fleet tools:
['cinebot_check_showtimes', 'cinebot_cancel_booking', 'cinebot_get_seat_map', 'errors_risky_lookup']

Using client group


In [88]:
from fastmcp.client import Client
from fastmcp.client.group import ClientGroup


In [94]:
group = ClientGroup(
    {
        "cinebot": Client(Path("cinebot_mcp_server.py")),
        "errors": Client(Path("cinebot_error_server.py"),mode="legacy"),
        'time_track_server':Client('https://time-track-mcp-server-mcpg-uys.vercel.app/mcp')
    }
)
async with MCPAdapter(group) as adapter:
    group_tools = await adapter.list_tools()
    print("ClientGroup fleet tools:", [t.name for t in group_tools])

ClientGroup fleet tools:
[
    'cinebot_check_showtimes',
    'cinebot_cancel_booking',
    'cinebot_get_seat_map',
    'errors_risky_lookup',
    'time_track_server_log_time',
    'time_track_server_get_timesheet',
    'time_track_server_get_project_summary',
    'time_track_server_list_projects'
]